# First transfer learning model (ResNet)

**Theme:**

*“Use a powerful brain, just change the decision layer.”*

## Imports

In [2]:
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader 
from torchvision import transforms, models, datasets 

torch.__version__

'2.8.0+cu129'

## Load Dataset and Transform

In [3]:
# Load transform 
transform = transforms.Compose([
    transforms.Resize((224,224)), # RestNet expect 224x224 pixels images
    transforms.ToTensor()
])


# Load dataset
train_data = datasets.CIFAR10(root='../week2/data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root='../week2/data', train=False, download=True, transform=transform) 

# Load in dataloader
train_loader = DataLoader(train_data, batch_size=64, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=64, shuffle=False) 

c:\Users\skroh\Desktop\Gen AI\.torchenv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
images, labels = next(iter(train_loader))
images.shape, labels.shape

(torch.Size([64, 3, 224, 224]), torch.Size([64]))

## Load Pretrained ResNet

In [6]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

This model already knows:

- Edges

- Textures

- Object parts

## Freeze Backbone

In [7]:
for param in model.parameters():
    param.requires_grad = False 

Freezed all the parameters

---

## Replace Final Layer

ResNet has 1000+ classes but <br>
CIFAR-10 need only 10 classes

In [8]:
model.fc = nn.Linear(model.fc.in_features, 10) 

Only the last layer learns

In [9]:
model.fc.in_features

512

## Training Set-up

In [10]:
# gpu set up 
device = None 
if torch.cuda.is_available():
    device = 'cuda:0' 
    print("GPU is available")
else:
    device = 'cpu'
    print("GPU is not available")

GPU is available


In [11]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) 
model = model.to(device)

Notice we only pass last layer parameters <br> 
Only the last layer should learn

## Training for 5 epochs

**Note : It took 8 mins to train in GPU** <br> 
Run the cell if you have time

In [12]:
model.train()
epochs = 5

for epoch in range(epochs):
    total_loss = 0.0  
    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        # forward pass 
        outputs = model(images) 
        # compute loss 
        loss = loss_fn(outputs,labels) 

        # zero grad
        optimizer.zero_grad() 
        # backward pass
        loss.backward() 
        # weights optimization 
        optimizer.step()

        total_loss += loss.item()
    
    print(f"Epoch: {epoch+1} | TrainLoss: {total_loss/len(train_loader):.6f}")


Epoch: 1 | TrainLoss: 0.841793
Epoch: 2 | TrainLoss: 0.620665
Epoch: 3 | TrainLoss: 0.595932
Epoch: 4 | TrainLoss: 0.580233
Epoch: 5 | TrainLoss: 0.572858


## Evaluation

In [13]:
model.eval()
eval_loss = 0.0 

with torch.no_grad():
    total = 0 
    correct = 0 
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images) 
        loss = loss_fn(outputs, labels) 

        predictions = torch.argmax(outputs,dim=1) 
        correct += (predictions==labels).sum().item() 
        total += len(labels)

        eval_loss += loss.item()
    
    print(f"TestLoss: {eval_loss/len(test_loader):.6f} | Accuracy: {correct/total * 100 :.2f} %")

    

TestLoss: 0.581244 | Accuracy: 80.21 %


# Interaction

### Key Confusion

We froze everything, trained only 512×10 parameters.
- Why is it still slow? (In MNIST, takes 3+ mins to train but now more than 3x)



---

Even if weights are frozen, the full network still runs in the forward pass.

Freezing only stops:

- Gradient computation

- Weight updates

It does NOT stop:

- Convolutions

- Feature extraction

- Forward propagation

---

### What Actually Happens During Training

Each batch still does:
```bash
Image → ResNet backbone (18 layers)

Feature maps computed

Passed to final FC layer

Loss computed

Backprop only through FC
```
So:

- We’re training few parameters

- BUT you’re computing through a huge network

That’s why it’s slower.

---

### Why Our Old CNN Was Faster


Compare complexity:

our old CNN:

- 2 conv layers

- Very small network

- Designed for MNIST

ResNet18:

- 18 layers

- Many filters

- Millions of operations per image

Even with frozen weights:

- Computation still happens

So:

- Computation cost ≠ number of trainable parameters

This is the key insight.

---

### Another BIG Reason: Image Size


Our old CNN:

MNIST → 28×28

ResNet:

- CIFAR resized → 224×224

Compare pixels:

- 28×28 = 784 pixels

- 224×224 = 50,176 pixels

That’s 64× more pixels per image.

- So each forward pass is much heavier.

This is the main reason for the slowdown.

---

### Why Accuracy is Only ~81%

Also normal.

Reason:

- ResNet trained on ImageNet (high-res natural images)

- CIFAR10 images are:

  - Tiny (32×32 originally)

- Low detail

- Different distribution

And:

- We only trained last layer

So performance is limited.

Next step later:

- Fine-tuning last few layers → accuracy jumps.

> Key Concept We Just Learned

Very important:

- Freezing reduces learning cost, not computation cost.
  
---

### Small Optimization Tip (Optional)


Instead of resizing to 224×224, try:
```java
transforms.Resize(128)
```
ResNet can still work.

This will:

- Speed up training

- Reduce GPU load